# Hierarchies and Graphs

## Quick notes
### What this covers
- Trees stored as an adjacency list (`id`, `parent_id`) and queried with recursive CTEs: descendants, ancestors, depth, paths, leaves, subtree totals, level by level.
- Cycle protection, reachability and shortest paths in a small graph, bill of materials (BOM) explosion with quantities.
- Three storage alternatives and their trade-offs: closure table (built and maintained with triggers), materialized path, nested set.
### Key points
- A recursive CTE has two parts joined by `UNION ALL`: an anchor (the starting rows) and a recursive step that reads the rows produced so far. It stops when the step returns no new rows.
- Going down: join `child.parent_id = cte.id`. Going up: join `parent.id = cte.parent_id`.
- Carry helper columns through the recursion: `depth`, a `path` string, a running `qty` or `cost`.
- A tree has one parent per node. A graph can have many edges and cycles, so you must guard against loops.
- Alternatives trade read speed against write cost: closure table reads fast, nested set reads fast but writes are painful, materialized path is simple.
### Gotchas (the part worth revising twice)
- A cycle in the data (A's parent is C, C's parent is B, B's parent is A) makes a recursive CTE run forever. Guard with a depth cap, a path check, or `UNION` instead of `UNION ALL`.
- `UNION ALL` keeps duplicates and is the normal choice. With `UNION`, rows that repeat are dropped, which also stops some cycles, but only if the rows are exactly identical (no changing `depth` column).
- In a DAG (a part used in several assemblies) the same node is reached by several paths. Summing without thinking double counts. In a BOM that is correct (quantities multiply); in a tree report it is not.
- No `ORDER BY` means no promised order. Sort by `path` for an indented tree, by `depth` for levels.
- SQLite does not check that `parent_id` forms a tree. A foreign key only checks the parent exists.
### When it matters
- Org charts, category menus and breadcrumbs, folders, comment threads, bills of materials, routes and dependencies.
### One-line recap example
- `WITH RECURSIVE sub(id) AS (SELECT 2 UNION ALL SELECT e.emp_id FROM employees e JOIN sub ON e.manager_id = sub.id) SELECT * FROM sub;` means "employee 2 and everyone below them, at any depth".

## 01. Seed data
- `employees`: a small org chart (`manager_id` points to the boss; the CEO has `NULL`). Managers are inserted before their reports, so the foreign key is always satisfied (as separate statements, the manager must exist first).
- `categories` + `products`: a shop menu tree. `parts` + `bom`: a bicycle bill of materials. `flights`: a directed graph with one cycle and one island.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, name TEXT NOT NULL,
    manager_id INTEGER REFERENCES employees(emp_id),
    title TEXT NOT NULL, salary INTEGER NOT NULL
);
INSERT INTO employees VALUES
    (1,'Ava',NULL,'CEO',250),(2,'Ben',1,'CTO',200),(3,'Cara',1,'CFO',190),
    (4,'Dev',2,'Eng Manager',150),(5,'Eli',2,'Data Manager',145),
    (6,'Fay',4,'Engineer',120),(7,'Gus',4,'Engineer',115),(8,'Hana',5,'Data Analyst',100),
    (9,'Ian',3,'Accountant',90),(10,'Jo',9,'Intern',40);
CREATE TABLE categories (cat_id INTEGER PRIMARY KEY, name TEXT NOT NULL, parent_id INTEGER REFERENCES categories(cat_id));
INSERT INTO categories VALUES
    (1,'Electronics',NULL),(2,'Phones',1),(3,'Smartphones',2),(4,'Feature Phones',2),
    (5,'Laptops',1),(6,'Gaming Laptops',5),(7,'Books',NULL),(8,'Fiction',7);
CREATE TABLE products (product_id INTEGER PRIMARY KEY, name TEXT NOT NULL, cat_id INTEGER NOT NULL REFERENCES categories);
INSERT INTO products VALUES (1,'Phone X',3),(2,'Brick 3310',4),(3,'Gamer 15',6),(4,'Ultrabook',5),(5,'Novel One',8);
CREATE TABLE parts (part TEXT PRIMARY KEY, unit_cost REAL);
INSERT INTO parts VALUES ('bike',NULL),('wheel',NULL),('frame',NULL),('handlebar',NULL),
    ('rim',20),('spoke',0.5),('tire',15),('tube',8),('weld_kit',5),('grip',3);
CREATE TABLE bom (parent TEXT NOT NULL REFERENCES parts, child TEXT NOT NULL REFERENCES parts, qty INTEGER NOT NULL,
                  PRIMARY KEY (parent, child));
INSERT INTO bom VALUES ('bike','wheel',2),('bike','frame',1),('bike','handlebar',1),
    ('wheel','rim',1),('wheel','spoke',36),('wheel','tire',1),
    ('frame','tube',3),('frame','weld_kit',1),('handlebar','tube',1),('handlebar','grip',2);
CREATE TABLE flights (src TEXT NOT NULL, dst TEXT NOT NULL, cost INTEGER NOT NULL, PRIMARY KEY (src, dst));
INSERT INTO flights VALUES
    ('AMS','LHR',100),('AMS','CDG',80),('CDG','LHR',60),('LHR','JFK',400),('CDG','JFK',450),
    ('LHR','DUB',90),('DUB','JFK',350),('JFK','SFO',300),('JFK','LHR',380),('OSL','BGO',70);
SELECT * FROM employees ORDER BY emp_id;

emp_id | name | manager_id | title        | salary
-------+------+------------+--------------+-------
1      | Ava  | NULL       | CEO          | 250   
2      | Ben  | 1          | CTO          | 200   
3      | Cara | 1          | CFO          | 190   
4      | Dev  | 2          | Eng Manager  | 150   
5      | Eli  | 2          | Data Manager | 145   
6      | Fay  | 4          | Engineer     | 120   
7      | Gus  | 4          | Engineer     | 115   
8      | Hana | 5          | Data Analyst | 100   
9      | Ian  | 3          | Accountant   | 90    
10     | Jo   | 9          | Intern       | 40    
(10 rows)



## 02. Anatomy of a recursive CTE
- Anchor: `SELECT 1`. Recursive step: `SELECT n + 1 FROM counter WHERE n < 5`. The `WHERE` is the stop condition.
- Each round feeds only the rows produced in the previous round back into the step.
- The same idea made the date spine in 04-Patterns/04-Time-Series-Patterns. For plain CTEs see 02-Querying/03-Subqueries-And-CTEs.

In [3]:
%%sql
WITH RECURSIVE counter(n) AS (
    SELECT 1
    UNION ALL
    SELECT n + 1 FROM counter WHERE n < 5
)
SELECT n, n * n AS square FROM counter ORDER BY n;

n | square
--+-------
1 | 1     
2 | 4     
3 | 9     
4 | 16    
5 | 25    
(5 rows)



## 03. All descendants of a node (with depth)
- Anchor: the starting employee (depth 0). Step: join employees whose `manager_id` is a row we already have, with `depth + 1`.
- This returns everyone in Ben's part of the company, at any depth.
- Compare with a plain self join (02-Querying/01-Joins): that only reaches a fixed number of levels.

In [4]:
%%sql
WITH RECURSIVE sub(emp_id, name, depth) AS (
    SELECT emp_id, name, 0 FROM employees WHERE emp_id = 2
    UNION ALL
    SELECT e.emp_id, e.name, sub.depth + 1
    FROM employees e JOIN sub ON e.manager_id = sub.emp_id
)
SELECT emp_id, name, depth FROM sub ORDER BY depth, name;

emp_id | name | depth
-------+------+------
2      | Ben  | 0    
4      | Dev  | 1    
5      | Eli  | 1    
6      | Fay  | 2    
7      | Gus  | 2    
8      | Hana | 2    
(6 rows)



## 04. All ancestors of a node (chain of command)
- Going up: anchor is the starting employee, step joins the row whose `emp_id` equals the current `manager_id`.
- It stops by itself at the CEO, because `manager_id` is `NULL` and the join finds nothing.
- `hops` counts how many levels above Hana each person is.

In [5]:
%%sql
WITH RECURSIVE up(emp_id, name, manager_id, hops) AS (
    SELECT emp_id, name, manager_id, 0 FROM employees WHERE name = 'Hana'
    UNION ALL
    SELECT e.emp_id, e.name, e.manager_id, up.hops + 1
    FROM employees e JOIN up ON e.emp_id = up.manager_id
)
SELECT hops, name FROM up ORDER BY hops;

hops | name
-----+-----
0    | Hana
1    | Eli 
2    | Ben 
3    | Ava 
(4 rows)



## 05. Depth, path string and an indented tree
- Carry a `path` column: `parent_path || ' > ' || name`. It is the breadcrumb for each node.
- Sorting by a path built from names gives depth-first order (a parent right before its children), which is what a printed tree needs.
- `substr('      ', 1, depth * 2)` or `replace(hex(zeroblob(depth * 2)), '00', ' ')` makes indentation.

In [6]:
%%sql
WITH RECURSIVE tree(emp_id, name, depth, path) AS (
    SELECT emp_id, name, 0, name FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, t.depth + 1, t.path || ' > ' || e.name
    FROM employees e JOIN tree t ON e.manager_id = t.emp_id
)
SELECT replace(hex(zeroblob(depth * 2)), '00', ' ') || name AS org_chart, depth, path
FROM tree ORDER BY path;

org_chart  | depth | path                  
-----------+-------+-----------------------
Ava        | 0     | Ava                   
  Ben      | 1     | Ava > Ben             
    Dev    | 2     | Ava > Ben > Dev       
      Fay  | 3     | Ava > Ben > Dev > Fay 
      Gus  | 3     | Ava > Ben > Dev > Gus 
    Eli    | 2     | Ava > Ben > Eli       
      Hana | 3     | Ava > Ben > Eli > Hana
  Cara     | 1     | Ava > Cara            
    Ian    | 2     | Ava > Cara > Ian      
      Jo   | 3     | Ava > Cara > Ian > Jo 
(10 rows)



## 06. Roots and leaves
- Root: no parent (`manager_id IS NULL`). Leaf: nobody has it as a parent. Check with `NOT EXISTS`.
- `NOT IN (SELECT manager_id ...)` is a trap: the subquery contains a `NULL` (the CEO's own manager), so `NOT IN` returns no rows at all. See 02-Querying/03-Subqueries-And-CTEs.

In [7]:
%%sql
SELECT e.name, e.title,
       CASE WHEN e.manager_id IS NULL THEN 'root' END AS is_root,
       CASE WHEN NOT EXISTS (SELECT 1 FROM employees c WHERE c.manager_id = e.emp_id) THEN 'leaf' END AS is_leaf
FROM employees e ORDER BY e.emp_id;
SELECT name FROM employees WHERE emp_id NOT IN (SELECT manager_id FROM employees) ORDER BY name;
SELECT name FROM employees WHERE emp_id NOT IN (SELECT manager_id FROM employees WHERE manager_id IS NOT NULL) ORDER BY name;

name | title        | is_root | is_leaf
-----+--------------+---------+--------
Ava  | CEO          | root    | NULL   
Ben  | CTO          | NULL    | NULL   
Cara | CFO          | NULL    | NULL   
Dev  | Eng Manager  | NULL    | NULL   
Eli  | Data Manager | NULL    | NULL   
Fay  | Engineer     | NULL    | leaf   
Gus  | Engineer     | NULL    | leaf   
Hana | Data Analyst | NULL    | leaf   
Ian  | Accountant   | NULL    | NULL   
Jo   | Intern       | NULL    | leaf   
(10 rows)

name
----
(0 rows)

name
----
Fay 
Gus 
Hana
Jo  
(4 rows)



- The first plain `NOT IN` returns 0 rows because of the `NULL`. With the `NULL` filtered out you get the 4 leaves (Fay, Gus, Hana, Jo), matching the `leaf` flags above.

## 07. Subtree sizes and rolled-up totals
- Idea: build a list of every (root, member) pair with a recursive CTE, where each node starts as its own root. Then `GROUP BY root`.
- `COUNT(*)` is the subtree size (including the node itself). `SUM(salary)` is the rolled-up payroll, like a department budget.
- The total for the CEO must equal the sum over everyone: a quick check that nothing was lost or double counted.

In [8]:
%%sql
WITH RECURSIVE reach(root, member) AS (
    SELECT emp_id, emp_id FROM employees
    UNION ALL
    SELECT r.root, e.emp_id FROM reach r JOIN employees e ON e.manager_id = r.member
)
SELECT p.name, p.title, COUNT(*) AS subtree_size, SUM(m.salary) AS rolled_up_salary
FROM reach r
JOIN employees p ON p.emp_id = r.root
JOIN employees m ON m.emp_id = r.member
GROUP BY p.emp_id ORDER BY rolled_up_salary DESC;
SELECT SUM(salary) AS everyone FROM employees;

name | title        | subtree_size | rolled_up_salary
-----+--------------+--------------+-----------------
Ava  | CEO          | 10           | 1400            
Ben  | CTO          | 6            | 830             
Dev  | Eng Manager  | 3            | 385             
Cara | CFO          | 3            | 320             
Eli  | Data Manager | 2            | 245             
Ian  | Accountant   | 2            | 130             
Fay  | Engineer     | 1            | 120             
Gus  | Engineer     | 1            | 115             
Hana | Data Analyst | 1            | 100             
Jo   | Intern       | 1            | 40              
(10 rows)

everyone
--------
1400    
(1 row)



## 08. Level by level (breadth-first output)
- Compute `depth` in the recursion, then group by it. Each group is one level of the org chart.
- `group_concat(x, ', ' ORDER BY x)` (SQLite 3.44+) lists the names of a level in a fixed order. Before 3.44 the order inside `group_concat` was not guaranteed.
- Useful for "who is at level 2?" and for finding the height of the tree (`MAX(depth)`).

In [9]:
%%sql
WITH RECURSIVE tree(emp_id, name, depth) AS (
    SELECT emp_id, name, 0 FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, t.depth + 1 FROM employees e JOIN tree t ON e.manager_id = t.emp_id
)
SELECT depth AS level, COUNT(*) AS people, group_concat(name, ', ' ORDER BY name) AS who
FROM tree GROUP BY depth ORDER BY depth;

level | people | who               
------+--------+-------------------
0     | 1      | Ava               
1     | 2      | Ben, Cara         
2     | 3      | Dev, Eli, Ian     
3     | 4      | Fay, Gus, Hana, Jo
(4 rows)



## 09. Cycle protection
- Bad data: A's parent is C, C's parent is B, B's parent is A. Walking up from A never reaches a root.
- Safety net 1: a depth cap (`WHERE hops < 6`). It stops the loop, but you still see repeated rows.
- Safety net 2: carry the visited ids in a `path` string and refuse to step onto a node already in it. This yields each node once.
- Safety net 3: `UNION` (not `UNION ALL`) without a changing column. Repeated identical rows are dropped, so the recursion ends.

In [10]:
%%sql
CREATE TABLE bad_cat (id INTEGER PRIMARY KEY, name TEXT NOT NULL, parent_id INTEGER);
INSERT INTO bad_cat VALUES (1,'A',3),(2,'B',1),(3,'C',2),(4,'R',NULL),(5,'S',4);
WITH RECURSIVE up(id, name, parent_id, hops) AS (
    SELECT id, name, parent_id, 0 FROM bad_cat WHERE id = 1
    UNION ALL
    SELECT c.id, c.name, c.parent_id, up.hops + 1 FROM bad_cat c JOIN up ON c.id = up.parent_id
    WHERE up.hops < 6
)
SELECT hops, name FROM up ORDER BY hops;

hops | name
-----+-----
0    | A   
1    | C   
2    | B   
3    | A   
4    | C   
5    | B   
6    | A   
(7 rows)



In [11]:
%%sql
WITH RECURSIVE up(id, name, parent_id, path) AS (
    SELECT id, name, parent_id, ',' || id || ',' FROM bad_cat WHERE id = 1
    UNION ALL
    SELECT c.id, c.name, c.parent_id, up.path || c.id || ','
    FROM bad_cat c JOIN up ON c.id = up.parent_id
    WHERE instr(up.path, ',' || c.id || ',') = 0
)
SELECT name, path FROM up ORDER BY length(path);
WITH RECURSIVE up(id, parent_id) AS (
    SELECT id, parent_id FROM bad_cat WHERE id = 1
    UNION
    SELECT c.id, c.parent_id FROM bad_cat c JOIN up ON c.id = up.parent_id
)
SELECT id FROM up ORDER BY id;

name | path   
-----+--------
A    | ,1,    
C    | ,1,3,  
B    | ,1,3,2,
(3 rows)

id
--
1 
2 
3 
(3 rows)



- Finding the broken rows: compute every (start, ancestor) pair (finite, so `UNION` ends) and look for a node that is its own ancestor.

In [12]:
%%sql
WITH RECURSIVE anc(start, id) AS (
    SELECT id, parent_id FROM bad_cat WHERE parent_id IS NOT NULL
    UNION
    SELECT a.start, c.parent_id FROM anc a JOIN bad_cat c ON c.id = a.id WHERE c.parent_id IS NOT NULL
)
SELECT b.id, b.name FROM bad_cat b WHERE EXISTS (SELECT 1 FROM anc WHERE anc.start = b.id AND anc.id = b.id) ORDER BY b.id;

id | name
---+-----
1  | A   
2  | B   
3  | C   
(3 rows)



## 10. Reachability and fewest-hops path in a graph
- A graph is a table of edges (`src`, `dst`). The flight network has a cycle (JFK and LHR fly to each other) and an island (OSL to BGO).
- Reachability: all airports you can get to from AMS. `UNION` removes repeats, so the cycle cannot loop.
- Shortest path by hops: carry the path and the hop count, forbid revisiting an airport, then pick the row with the fewest hops (the first time a node is reached in breadth-first order is also its shortest).

In [13]:
%%sql
WITH RECURSIVE reach(airport) AS (
    SELECT 'AMS'
    UNION
    SELECT f.dst FROM flights f JOIN reach r ON f.src = r.airport
)
SELECT airport FROM reach ORDER BY airport;

airport
-------
AMS    
CDG    
DUB    
JFK    
LHR    
SFO    
(6 rows)



In [14]:
%%sql
WITH RECURSIVE route(airport, hops, path) AS (
    SELECT 'AMS', 0, 'AMS'
    UNION ALL
    SELECT f.dst, r.hops + 1, r.path || ' > ' || f.dst
    FROM route r JOIN flights f ON f.src = r.airport
    WHERE instr(r.path, f.dst) = 0
)
SELECT airport, MIN(hops) AS fewest_hops, path
FROM route GROUP BY airport ORDER BY fewest_hops, airport;

airport | fewest_hops | path                 
--------+-------------+----------------------
AMS     | 0           | AMS                  
CDG     | 1           | AMS > CDG            
LHR     | 1           | AMS > LHR            
DUB     | 2           | AMS > LHR > DUB      
JFK     | 2           | AMS > CDG > JFK      
SFO     | 3           | AMS > CDG > JFK > SFO
(6 rows)



- `GROUP BY airport` with a single `MIN()` returns the `path` from the row that has the minimum (a SQLite feature, see 02-Querying/02-Aggregation-And-Grouping). When two routes tie on hops (JFK and SFO can each be reached via LHR or via CDG), the shown path is one of them. Do not rely on which one.

## 11. Cheapest route (weighted edges)
- Same recursion, but carry `total_cost` and compare costs instead of hops.
- Enumerating all simple paths grows quickly with graph size. This is fine for a small graph like this one, not for a road network. Use a graph library or database for that.
- Add `ORDER BY total_cost, path` and `LIMIT` for a deterministic top N.

In [15]:
%%sql
WITH RECURSIVE route(airport, total_cost, path) AS (
    SELECT 'AMS', 0, 'AMS'
    UNION ALL
    SELECT f.dst, r.total_cost + f.cost, r.path || ' > ' || f.dst
    FROM route r JOIN flights f ON f.src = r.airport
    WHERE instr(r.path, f.dst) = 0
)
SELECT total_cost, path FROM route WHERE airport = 'SFO' ORDER BY total_cost, path;

total_cost | path                             
-----------+----------------------------------
800        | AMS > LHR > JFK > SFO            
830        | AMS > CDG > JFK > SFO            
840        | AMS > CDG > LHR > JFK > SFO      
840        | AMS > LHR > DUB > JFK > SFO      
880        | AMS > CDG > LHR > DUB > JFK > SFO
(5 rows)



- Five ways to SFO. The cheapest is AMS > LHR > JFK > SFO at 800. The fewest-hop routes (3 hops) cost 800 and 830, so fewest hops is not the same as cheapest.

## 12. Bill of materials explosion with quantities
- A BOM is a tree where the same part can appear under several parents (a DAG). Quantities multiply down the levels: 2 wheels x 36 spokes = 72 spokes.
- Recursive step: `r.qty * b.qty`. At the end, `GROUP BY` the base part (no children) and `SUM(qty)` to add up the same part used in several places (tube is in the frame and the handlebar).
- Joining `parts` gives a rolled-up cost. Assemblies have `NULL` unit cost and are excluded by only keeping leaf parts.

In [16]:
%%sql
WITH RECURSIVE explode(part, qty, depth) AS (
    SELECT 'bike', 1, 0
    UNION ALL
    SELECT b.child, e.qty * b.qty, e.depth + 1 FROM explode e JOIN bom b ON b.parent = e.part
)
SELECT e.part, SUM(e.qty) AS total_qty, p.unit_cost, SUM(e.qty) * p.unit_cost AS line_cost
FROM explode e JOIN parts p ON p.part = e.part
WHERE NOT EXISTS (SELECT 1 FROM bom b WHERE b.parent = e.part)
GROUP BY e.part ORDER BY e.part;

part     | total_qty | unit_cost | line_cost
---------+-----------+-----------+----------
grip     | 2         | 3.0       | 6.0      
rim      | 2         | 20.0      | 40.0     
spoke    | 72        | 0.5       | 36.0     
tire     | 2         | 15.0      | 30.0     
tube     | 4         | 8.0       | 32.0     
weld_kit | 1         | 5.0       | 5.0      
(6 rows)



In [17]:
%%sql
WITH RECURSIVE explode(part, qty) AS (
    SELECT 'bike', 1
    UNION ALL
    SELECT b.child, e.qty * b.qty FROM explode e JOIN bom b ON b.parent = e.part
)
SELECT SUM(e.qty * p.unit_cost) AS total_bike_cost
FROM explode e JOIN parts p ON p.part = e.part WHERE p.unit_cost IS NOT NULL;

total_bike_cost
---------------
149.0          
(1 row)



- "Where used" is the same query going up: start from `tube` and join `b.child = e.part`, selecting `b.parent`.

## 13. Category breadcrumbs
- A breadcrumb is the path from the root to a node: `Electronics > Phones > Smartphones`.
- Build it top-down for all categories at once, then join products to it.
- Same idea gives "all products under Electronics, including subcategories": filter on the path prefix or on the ancestor.

In [18]:
%%sql
CREATE VIEW cat_paths AS
WITH RECURSIVE t(cat_id, name, depth, breadcrumb) AS (
    SELECT cat_id, name, 0, name FROM categories WHERE parent_id IS NULL
    UNION ALL
    SELECT c.cat_id, c.name, t.depth + 1, t.breadcrumb || ' > ' || c.name
    FROM categories c JOIN t ON c.parent_id = t.cat_id
)
SELECT cat_id, name, depth, breadcrumb FROM t;
SELECT p.name AS product, c.breadcrumb FROM products p JOIN cat_paths c ON c.cat_id = p.cat_id ORDER BY p.name;
SELECT p.name AS product FROM products p JOIN cat_paths c ON c.cat_id = p.cat_id
WHERE c.breadcrumb = 'Electronics' OR c.breadcrumb LIKE 'Electronics > %' ORDER BY p.name;

product    | breadcrumb                            
-----------+---------------------------------------
Brick 3310 | Electronics > Phones > Feature Phones 
Gamer 15   | Electronics > Laptops > Gaming Laptops
Novel One  | Books > Fiction                       
Phone X    | Electronics > Phones > Smartphones    
Ultrabook  | Electronics > Laptops                 
(5 rows)

product   
----------
Brick 3310
Gamer 15  
Phone X   
Ultrabook 
(4 rows)



## 14. Org chart reports: span of control and skip-level
- Span of control: how many direct reports each manager has. A plain `GROUP BY manager_id`, no recursion needed.
- Skip-level: people exactly two levels below. Self join twice, or filter the recursive result on `depth = 2`.
- Whole-company view: manager name next to each person, with the CEO shown as "(none)" via `LEFT JOIN` and `COALESCE`.

In [19]:
%%sql
SELECT m.name AS manager, COUNT(*) AS direct_reports
FROM employees e JOIN employees m ON m.emp_id = e.manager_id
GROUP BY m.emp_id ORDER BY direct_reports DESC, manager;
SELECT g.name AS skip_level_report, m.name AS via, b.name AS boss
FROM employees g
JOIN employees m ON m.emp_id = g.manager_id
JOIN employees b ON b.emp_id = m.manager_id
WHERE b.name = 'Ava' ORDER BY g.name;
SELECT e.name, e.title, COALESCE(m.name, '(none)') AS reports_to
FROM employees e LEFT JOIN employees m ON m.emp_id = e.manager_id ORDER BY e.emp_id;

manager | direct_reports
--------+---------------
Ava     | 2             
Ben     | 2             
Dev     | 2             
Cara    | 1             
Eli     | 1             
Ian     | 1             
(6 rows)

skip_level_report | via  | boss
------------------+------+-----
Dev               | Ben  | Ava 
Eli               | Ben  | Ava 
Ian               | Cara | Ava 
(3 rows)

name | title        | reports_to
-----+--------------+-----------
Ava  | CEO          | (none)    
Ben  | CTO          | Ava       
Cara | CFO          | Ava       
Dev  | Eng Manager  | Ben       
Eli  | Data Manager | Ben       
Fay  | Engineer     | Dev       
Gus  | Engineer     | Dev       
Hana | Data Analyst | Eli       
Ian  | Accountant   | Cara      
Jo   | Intern       | Ian       
(10 rows)



## 15. Closure table: build and query
- Recursive queries on every page view get slow on big trees. A closure table stores every (ancestor, descendant) pair, with the distance between them. Every node is also its own ancestor at depth 0.
- Build it once from the adjacency list with a recursive CTE. After that, no recursion is needed: descendants, ancestors and subtree sizes become plain joins.
- Cost: storage (about N x depth rows) and the need to keep it in sync.

In [20]:
%%sql
CREATE TABLE emp_closure (
    ancestor   INTEGER NOT NULL REFERENCES employees(emp_id) ON DELETE CASCADE,
    descendant INTEGER NOT NULL REFERENCES employees(emp_id) ON DELETE CASCADE,
    depth      INTEGER NOT NULL,
    PRIMARY KEY (ancestor, descendant)
);
WITH RECURSIVE c(a, d, depth) AS (
    SELECT emp_id, emp_id, 0 FROM employees
    UNION ALL
    SELECT c.a, e.emp_id, c.depth + 1 FROM c JOIN employees e ON e.manager_id = c.d
)
INSERT INTO emp_closure SELECT a, d, depth FROM c;
SELECT COUNT(*) AS closure_rows FROM emp_closure;

closure_rows
------------
30          
(1 row)



In [21]:
%%sql
SELECT 'below Ben' AS question, e.name, c.depth
FROM emp_closure c JOIN employees e ON e.emp_id = c.descendant
WHERE c.ancestor = 2 AND c.depth > 0
UNION ALL
SELECT 'above Hana', e.name, c.depth
FROM emp_closure c JOIN employees e ON e.emp_id = c.ancestor
WHERE c.descendant = 8 AND c.depth > 0
ORDER BY 1, 3, 2;
SELECT e.name, COUNT(*) AS subtree_size, SUM(d.salary) AS rolled_up_salary
FROM emp_closure c JOIN employees e ON e.emp_id = c.ancestor JOIN employees d ON d.emp_id = c.descendant
GROUP BY c.ancestor ORDER BY rolled_up_salary DESC;

question   | name | depth
-----------+------+------
above Hana | Eli  | 1    
above Hana | Ben  | 2    
above Hana | Ava  | 3    
below Ben  | Dev  | 1    
below Ben  | Eli  | 1    
below Ben  | Fay  | 2    
below Ben  | Gus  | 2    
below Ben  | Hana | 2    
(8 rows)

name | subtree_size | rolled_up_salary
-----+--------------+-----------------
Ava  | 10           | 1400            
Ben  | 6            | 830             
Dev  | 3            | 385             
Cara | 3            | 320             
Eli  | 2            | 245             
Ian  | 2            | 130             
Fay  | 1            | 120             
Gus  | 1            | 115             
Hana | 1            | 100             
Jo   | 1            | 40              
(10 rows)



- The closure table has 30 rows for 10 employees (the sum of every node's depth + 1). The last query gives the same numbers as section 07, without any recursion.

## 16. Keeping the closure table in sync with triggers
- Three triggers keep `emp_closure` correct when `employees` changes (see 05-DDL-And-Integrity for trigger basics):
- Insert: add a self row, plus one row per ancestor of the new manager with `depth + 1`.
- Move (update `manager_id`): remove the links between the moved subtree and its old ancestors, then add links between the subtree and the new ancestors.
- Guard: refuse a move that puts a node under its own descendant, which would create a cycle.
- Delete: the `ON DELETE CASCADE` on the closure table removes the rows for you.

In [22]:
%%sql
CREATE TRIGGER emp_ai AFTER INSERT ON employees
BEGIN
    INSERT INTO emp_closure (ancestor, descendant, depth) VALUES (NEW.emp_id, NEW.emp_id, 0);
    INSERT INTO emp_closure (ancestor, descendant, depth)
    SELECT ancestor, NEW.emp_id, depth + 1 FROM emp_closure WHERE descendant = NEW.manager_id;
END;
CREATE TRIGGER emp_guard BEFORE UPDATE OF manager_id ON employees
WHEN NEW.manager_id IS NOT NULL
 AND EXISTS (SELECT 1 FROM emp_closure WHERE ancestor = NEW.emp_id AND descendant = NEW.manager_id)
BEGIN
    SELECT RAISE(ABORT, 'cycle: the new manager is inside this subtree');
END;
CREATE TRIGGER emp_au AFTER UPDATE OF manager_id ON employees
BEGIN
    DELETE FROM emp_closure
    WHERE descendant IN (SELECT descendant FROM emp_closure WHERE ancestor = NEW.emp_id)
      AND ancestor IN (SELECT ancestor FROM emp_closure WHERE descendant = NEW.emp_id AND ancestor <> NEW.emp_id);
    INSERT INTO emp_closure (ancestor, descendant, depth)
    SELECT sup.ancestor, sub.descendant, sup.depth + sub.depth + 1
    FROM emp_closure sup, emp_closure sub
    WHERE sup.descendant = NEW.manager_id AND sub.ancestor = NEW.emp_id;
END;

- Hire Kim under Fay. Kim should get 5 closure rows: herself, Fay, Dev, Ben and Ava.

In [23]:
%%sql
INSERT INTO employees VALUES (11, 'Kim', 6, 'Engineer', 105);
SELECT a.name AS ancestor, c.depth FROM emp_closure c JOIN employees a ON a.emp_id = c.ancestor
WHERE c.descendant = 11 ORDER BY c.depth;

ancestor | depth
---------+------
Kim      | 0    
Fay      | 1    
Dev      | 2    
Ben      | 3    
Ava      | 4    
(5 rows)



- Move Dev (with Fay, Gus and Kim below) from Ben to Cara. Ben's subtree shrinks, Cara's grows, and nothing else changes.

In [24]:
%%sql
UPDATE employees SET manager_id = 3 WHERE emp_id = 4;
SELECT a.name AS manager, COUNT(*) - 1 AS people_below
FROM emp_closure c JOIN employees a ON a.emp_id = c.ancestor
WHERE a.name IN ('Ava', 'Ben', 'Cara', 'Dev') GROUP BY c.ancestor ORDER BY a.name;
SELECT a.name AS ancestor_of_Kim, c.depth FROM emp_closure c JOIN employees a ON a.emp_id = c.ancestor
WHERE c.descendant = 11 ORDER BY c.depth;

manager | people_below
--------+-------------
Ava     | 10          
Ben     | 2           
Cara    | 6           
Dev     | 3           
(4 rows)

ancestor_of_Kim | depth
----------------+------
Kim             | 0    
Fay             | 1    
Dev             | 2    
Cara            | 3    
Ava             | 4    
(5 rows)



- Now the guard: making Ava report to Hana would create a loop. Then move Dev back to Ben and remove Kim (the cascade cleans the closure rows).

In [25]:
%%sql expect_error
UPDATE employees SET manager_id = 8 WHERE emp_id = 1;

ERROR: cycle: the new manager is inside this subtree



In [26]:
%%sql
UPDATE employees SET manager_id = 2 WHERE emp_id = 4;
DELETE FROM employees WHERE emp_id = 11;
SELECT COUNT(*) AS closure_rows, (SELECT COUNT(*) FROM employees) AS employees FROM emp_closure;

closure_rows | employees
-------------+----------
30           | 10       
(1 row)



- Back to 30 closure rows for 10 employees. Deleting a node that still has children would fail on the `manager_id` foreign key (see 05-DDL-And-Integrity), so delete leaves first or re-parent their children.

## 17. Materialized path
- Store the whole path in one column on the row itself: `'/1/2/5/8/'` for Hana (the ids from the root down to her, with a slash at both ends).
- Descendants of Ben: `path LIKE '/1/2/%'`. Ancestors of Hana: every path that is a prefix of hers. Depth: count the slashes and subtract 1 (the CEO's path `'/1/'` has 2 slashes, so it is depth 1, as in the query below; subtract 2 for a 0-based depth). No joins, no recursion, one extra column.
- Always end the path with a slash. Without it, `'/1/2%'` also matches `'/1/20/'`.
- Moving a subtree means rewriting the path text of every node below it with `UPDATE ... SET path = new_prefix || substr(path, length(old_prefix) + 1)`.

In [27]:
%%sql
ALTER TABLE employees ADD COLUMN path TEXT;
WITH RECURSIVE p(emp_id, path) AS (
    SELECT emp_id, '/' || emp_id || '/' FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, p.path || e.emp_id || '/' FROM employees e JOIN p ON e.manager_id = p.emp_id
)
UPDATE employees SET path = (SELECT p.path FROM p WHERE p.emp_id = employees.emp_id);
SELECT emp_id, name, path FROM employees ORDER BY path;

emp_id | name | path      
-------+------+-----------
1      | Ava  | /1/       
2      | Ben  | /1/2/     
4      | Dev  | /1/2/4/   
6      | Fay  | /1/2/4/6/ 
7      | Gus  | /1/2/4/7/ 
5      | Eli  | /1/2/5/   
8      | Hana | /1/2/5/8/ 
3      | Cara | /1/3/     
9      | Ian  | /1/3/9/   
10     | Jo   | /1/3/9/10/
(10 rows)



In [28]:
%%sql
SELECT 'below Ben' AS question, name, length(path) - length(replace(path, '/', '')) - 1 AS depth
FROM employees WHERE path LIKE '/1/2/%' AND emp_id <> 2
UNION ALL
SELECT 'above Hana', a.name, length(a.path) - length(replace(a.path, '/', '')) - 1
FROM employees a, employees h
WHERE h.name = 'Hana' AND h.path LIKE a.path || '%' AND a.emp_id <> h.emp_id
ORDER BY 1, 3, 2;
SELECT '/1/20/' LIKE '/1/2%' AS without_trailing_slash_wrong, '/1/20/' LIKE '/1/2/%' AS with_trailing_slash_right;

question   | name | depth
-----------+------+------
above Hana | Ava  | 1    
above Hana | Ben  | 2    
above Hana | Eli  | 3    
below Ben  | Dev  | 3    
below Ben  | Eli  | 3    
below Ben  | Fay  | 4    
below Ben  | Gus  | 4    
below Ben  | Hana | 4    
(8 rows)

without_trailing_slash_wrong | with_trailing_slash_right
-----------------------------+--------------------------
1                            | 0                        
(1 row)



## 18. Nested set (left and right numbers)
- Walk the tree depth-first and number each node twice: `lft` when you enter it, `rgt` when you leave it. A node's descendants are exactly the rows whose `lft` lies between its `lft` and `rgt`.
- Reads are very fast: subtree = `BETWEEN`. Subtree size = `(rgt - lft + 1) / 2`. Leaf = `rgt = lft + 1`.
- Writes are expensive: adding one node shifts the numbers of many rows. It is a good fit for trees that are read often and rarely change (a product taxonomy).
- Below the numbers are computed with pre-order position and subtree size: `lft = 2 * pre - depth - 1`, `rgt = lft + 2 * size - 1`.

In [29]:
%%sql
CREATE TABLE emp_nested AS
WITH RECURSIVE t(emp_id, name, depth, sortpath) AS (
    SELECT emp_id, name, 0, name FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.emp_id, e.name, t.depth + 1, t.sortpath || '/' || e.name
    FROM employees e JOIN t ON e.manager_id = t.emp_id
), o AS (
    SELECT emp_id, name, depth, ROW_NUMBER() OVER (ORDER BY sortpath) AS pre FROM t
)
SELECT o.emp_id, o.name,
       2 * o.pre - o.depth - 1 AS lft,
       2 * o.pre - o.depth - 1 + 2 * (SELECT COUNT(*) FROM emp_closure c WHERE c.ancestor = o.emp_id) - 1 AS rgt
FROM o;
SELECT * FROM emp_nested ORDER BY lft;

emp_id | name | lft | rgt
-------+------+-----+----
1      | Ava  | 1   | 20 
2      | Ben  | 2   | 13 
4      | Dev  | 3   | 8  
6      | Fay  | 4   | 5  
7      | Gus  | 6   | 7  
5      | Eli  | 9   | 12 
8      | Hana | 10  | 11 
3      | Cara | 14  | 19 
9      | Ian  | 15  | 18 
10     | Jo   | 16  | 17 
(10 rows)



In [30]:
%%sql
SELECT 'below Ben' AS question, d.name
FROM emp_nested b JOIN emp_nested d ON d.lft > b.lft AND d.rgt < b.rgt WHERE b.name = 'Ben'
UNION ALL
SELECT 'above Hana', a.name
FROM emp_nested h JOIN emp_nested a ON h.lft > a.lft AND h.rgt < a.rgt WHERE h.name = 'Hana'
ORDER BY 1, 2;
SELECT name, (rgt - lft + 1) / 2 AS subtree_size, CASE WHEN rgt = lft + 1 THEN 'leaf' END AS is_leaf
FROM emp_nested ORDER BY lft;

question   | name
-----------+-----
above Hana | Ava 
above Hana | Ben 
above Hana | Eli 
below Ben  | Dev 
below Ben  | Eli 
below Ben  | Fay 
below Ben  | Gus 
below Ben  | Hana
(8 rows)

name | subtree_size | is_leaf
-----+--------------+--------
Ava  | 10           | NULL   
Ben  | 6            | NULL   
Dev  | 3            | NULL   
Fay  | 1            | leaf   
Gus  | 1            | leaf   
Eli  | 2            | NULL   
Hana | 1            | leaf   
Cara | 3            | NULL   
Ian  | 2            | NULL   
Jo   | 1            | leaf   
(10 rows)



- What an insert costs: adding a new engineer under Fay (a leaf) means opening a gap of 2 numbers at Fay's `rgt`. The transaction below counts the rows touched, then rolls back. Of 10 rows, 10 get a new `rgt` and 6 get a new `lft`: nearly the whole table for one new leaf.

In [31]:
%%sql
BEGIN;
UPDATE emp_nested SET rgt = rgt + 2 WHERE rgt >= (SELECT rgt FROM emp_nested WHERE name = 'Fay');
SELECT changes() AS rows_shifted_by_rgt_update;
UPDATE emp_nested SET lft = lft + 2 WHERE lft > (SELECT rgt FROM emp_nested WHERE name = 'Fay') - 2;
SELECT changes() AS rows_shifted_by_lft_update;
ROLLBACK;
SELECT COUNT(*) AS rows_after_rollback FROM emp_nested;

rows_shifted_by_rgt_update
--------------------------
10                        
(1 row)

rows_shifted_by_lft_update
--------------------------
6                         
(1 row)

rows_after_rollback
-------------------
10                 
(1 row)



## 19. Choosing a design
- Adjacency list (`parent_id`): simplest, easy to write, needs a recursive CTE to read a subtree.
- Closure table: fast reads of any subtree or chain with plain joins, easy moves with triggers, extra table.
- Materialized path: one extra column, fast subtree reads with `LIKE`, moves rewrite many strings, no foreign key protection for the path.
- Nested set: fastest subtree reads and counts, painful writes.
- Rule of thumb: start with the adjacency list. Reach for a closure table when reads dominate and queries get slow. Use nested set only for almost-static trees.

In [32]:
%%sql
WITH t(design, read_subtree, read_ancestors, insert_leaf, move_subtree, integrity) AS (VALUES
    ('adjacency list', 'recursive CTE', 'recursive CTE', 'one row', 'one row', 'foreign key (cycles possible)'),
    ('closure table', 'one join', 'one join', 'a few rows (trigger)', 'many rows (trigger)', 'foreign keys on both columns'),
    ('materialized path', 'LIKE prefix', 'prefix match', 'one row', 'rewrite subtree paths', 'none for the path text'),
    ('nested set', 'BETWEEN', 'BETWEEN', 'shifts many rows', 'shifts many rows', 'none (numbers can drift)')
)
SELECT * FROM t ORDER BY design;

design            | read_subtree  | read_ancestors | insert_leaf          | move_subtree          | integrity                    
------------------+---------------+----------------+----------------------+-----------------------+------------------------------
adjacency list    | recursive CTE | recursive CTE  | one row              | one row               | foreign key (cycles possible)
closure table     | one join      | one join       | a few rows (trigger) | many rows (trigger)   | foreign keys on both columns 
materialized path | LIKE prefix   | prefix match   | one row              | rewrite subtree paths | none for the path text       
nested set        | BETWEEN       | BETWEEN        | shifts many rows     | shifts many rows      | none (numbers can drift)     
(4 rows)

